# SLAVA v0: collection and review of 102 simulation scenes

Этот notebook собирает environment-first inventory:

- 30 LIBERO tasks × init states `[0, 17, 34]` = 90 сцен;
- 4 SimplerEnv bridge tasks × episode IDs `[0, 8, 16]` = 12 сцен;
- всего 102 строки `task × init_state`.

Симуляторы запускаются в отдельных virtual environments. Notebook хранит объединённый DataFrame, интерактивную разметку и экспорт JSONL/CSV. Сбор работает в resume-режиме.

In [35]:
from pathlib import Path
import json
import importlib
import os
import shutil
import subprocess
import sys
import pandas as pd
from IPython.display import display

In [36]:


def find_project_root():
    candidates = [os.environ.get('SLAVA_ROOT'), Path.cwd(), Path.cwd().parent]
    for candidate in candidates:
        if not candidate:
            continue
        candidate = Path(candidate).expanduser().resolve()
        if (candidate / 'src' / 'slava_inventory').is_dir():
            return candidate
    raise RuntimeError('Не найден корень SLAVA_dev. Откройте папку проекта в VS Code перед запуском ноутбука.')

PROJECT_ROOT = find_project_root()
DEPS_DIR = Path(os.environ.get('SLAVA_DEPS_DIR', PROJECT_ROOT.parent)).expanduser().resolve()
LIBERO_REPO = Path(os.environ.get('LIBERO_ROOT', DEPS_DIR / 'LIBERO')).expanduser().resolve()
SIMPLER_REPO = Path(os.environ.get('SIMPLERENV_ROOT', DEPS_DIR / 'SimplerEnv')).expanduser().resolve()
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import slava_inventory.notebook_ui as notebook_ui
notebook_ui = importlib.reload(notebook_ui)
InventoryReviewer = notebook_ui.InventoryReviewer
LexiconReviewer = notebook_ui.LexiconReviewer
VisibilityReviewer = notebook_ui.VisibilityReviewer
create_or_update_lexicon = notebook_ui.create_or_update_lexicon
export_inventory_dataframe = notebook_ui.export_inventory_dataframe
merge_inventories = notebook_ui.merge_inventories

conda_candidates = [
    os.environ.get('CONDA_EXE'),
    shutil.which('conda'),
    '/opt/miniforge3/bin/conda',
    '/opt/conda/bin/conda',
]
CONDA_EXE = next((str(Path(p)) for p in conda_candidates if p and Path(p).is_file()), None)
LIBERO_CONDA_ENV = 'slava-libero'
SIMPLER_CONDA_ENV = 'slava-simpler'
print('Project:', PROJECT_ROOT)
print('Data:', DATA_DIR)
print('Conda:', CONDA_EXE or 'NOT FOUND')

Project: /Users/alexkarachun/Documents/DEV/SLAVA_dev
Data: /Users/alexkarachun/Documents/DEV/SLAVA_dev/data
Conda: /opt/miniconda3/bin/conda


## 1. Проверка путей и окружений

Если Conda environment отсутствует, сначала выполните инструкции из `README.md`.

In [37]:
# assert PROJECT_ROOT.exists(), PROJECT_ROOT
# assert LIBERO_REPO.exists(), LIBERO_REPO
# assert SIMPLER_REPO.exists(), SIMPLER_REPO

if CONDA_EXE is None:
    print('WARNING: Conda executable was not found. See README.md.')
    conda_envs = {}
else:
    result = subprocess.run(
        [CONDA_EXE, 'env', 'list', '--json'], capture_output=True, text=True, check=True
    )
    env_paths = json.loads(result.stdout)['envs']
    conda_envs = {Path(path).name: path for path in env_paths}
    display(pd.DataFrame(sorted(conda_envs.items()), columns=['environment', 'path']))

for env_name in [LIBERO_CONDA_ENV, SIMPLER_CONDA_ENV]:
    if env_name not in conda_envs:
        print(f'WARNING: Conda environment {env_name!r} was not found')

,environment,path
0,base,/opt/homebrew/Caskroom/miniconda/base
1,miniconda3,/opt/miniconda3
2,selfmade-diffusion,/opt/miniconda3/envs/selfmade-diffusion
3,through_guidance,/opt/miniconda3/envs/through_guidance


## 2. Сбор сцен

По умолчанию collectors не запускаются. Поменяйте нужный флаг на `True`.

- Повторный запуск безопасен: существующие `task_uid` пропускаются.
- `OVERWRITE_EXISTING=True` полностью пересоздаст соответствующий partial inventory.
- Ошибки отдельных сцен сохраняются в `data/collection_errors.jsonl`.
- Для LIBERO headless renderer запускается с `MUJOCO_GL=egl`.

In [38]:
RUN_LIBERO = False
RUN_SIMPLER = False
OVERWRITE_EXISTING = False
FAIL_FAST = True

def run_streaming(command, extra_env=None):
    env = None
    if extra_env:
        import os
        env = os.environ.copy()
        env.update(extra_env)
    print(' '.join(map(str, command)))
    subprocess.run(list(map(str, command)), cwd=PROJECT_ROOT, env=env, check=True)

if (RUN_LIBERO or RUN_SIMPLER) and CONDA_EXE is None:
    raise RuntimeError('Conda executable was not found. See README.md.')

if RUN_LIBERO:
    cmd = [
        CONDA_EXE, 'run', '--no-capture-output', '-n', LIBERO_CONDA_ENV,
        'python', PROJECT_ROOT / 'scripts/collect_libero.py',
        '--libero-repo', LIBERO_REPO,
        '--output-root', DATA_DIR,
        '--init-state-ids', 0, 17, 34,
        '--image-size', 256,
        '--settle-steps', 0,
    ]
    if OVERWRITE_EXISTING:
        cmd.append('--overwrite')
    if FAIL_FAST:
        cmd.append('--fail-fast')
    run_streaming(cmd, {'MUJOCO_GL': 'egl', 'MUJOCO_EGL_DEVICE_ID': '0'})

if RUN_SIMPLER:
    cmd = [
        CONDA_EXE, 'run', '--no-capture-output', '-n', SIMPLER_CONDA_ENV,
        'python', PROJECT_ROOT / 'scripts/collect_simpler.py',
        '--simpler-repo', SIMPLER_REPO,
        '--output-root', DATA_DIR,
        '--episode-ids', 0, 8, 16,
    ]
    if OVERWRITE_EXISTING:
        cmd.append('--overwrite')
    if FAIL_FAST:
        cmd.append('--fail-fast')
    run_streaming(cmd)

## 3. Объединение partial inventories

При повторном объединении ручные annotations из существующего `task_inventory.jsonl` сохраняются.

In [39]:
inventory_df = merge_inventories(DATA_DIR)
print('Total scenes:', len(inventory_df))
if inventory_df.empty:
    print('Inventory is empty. Set RUN_LIBERO/RUN_SIMPLER=True and run the collection cell.')
else:
    display(inventory_df.groupby('suite').size().rename('scenes').to_frame())
    display(inventory_df[['task_uid', 'suite', 'canonical_en', 'usable_for_slava']].head())
    if len(inventory_df) != 102:
        print('WARNING: expected 102 rows. Inspect data/collection_errors.jsonl and rerun collectors.')

Total scenes: 102


,scenes
suite,
libero_goal,30
libero_object,30
libero_spatial,30
simpler_bridge,12


,task_uid,suite,canonical_en,usable_for_slava
0,libero_goal__open_the_middle_drawer_of_the_cab...,libero_goal,open the middle drawer of the cabinet,None
1,libero_goal__open_the_middle_drawer_of_the_cab...,libero_goal,open the middle drawer of the cabinet,None
2,libero_goal__open_the_middle_drawer_of_the_cab...,libero_goal,open the middle drawer of the cabinet,None
3,libero_goal__open_the_top_drawer_and_put_the_b...,libero_goal,open the top drawer and put the bowl inside,None
4,libero_goal__open_the_top_drawer_and_put_the_b...,libero_goal,open the top drawer and put the bowl inside,None


## 4. Быстрая проверка видимости benchmark-объектов

Форма показывает одну сцену, обе доступные камеры и все объекты из канонического `objects_raw`.

Для каждого объекта выберите `visible`, `visible_partial` или `not visible`. `visible_partial` означает, что объект виден лишь частично, но его всё ещё можно распознать. У SimplerEnv нет wrist camera, поэтому `visible_wrist` остаётся `null`. Кнопка **Save + next** сразу атомарно сохраняет весь DataFrame в `data/task_inventory.jsonl` и открывает следующую незаполненную сцену. `?` означает, что поле ещё не проверено; такая сцена остаётся в фильтре **Only unfinished**. Кнопки **All visible** ускоряют разметку сцен, где все объекты хорошо видны.

In [40]:
visibility_reviewer = None
if inventory_df.empty:
    print('Visibility review is unavailable until the inventory has been collected.')
else:
    visibility_reviewer = VisibilityReviewer(inventory_df, DATA_DIR)
    visibility_reviewer.show()

visible - целевую зону объекта видно целиком

partial - видно только часть объекта (возможно целевую зону объекта не видно)

not visible - объекта нет на изображении


Сводка прогресса (можно выполнять повторно в любой момент):

In [41]:
if visibility_reviewer is None:
    print('No visibility-review summary yet.')
else:
    inventory_df = visibility_reviewer.df
    finished = sum(visibility_reviewer._scene_complete(i) for i in range(len(inventory_df)))
    print(f'Visibility finished: {finished}/{len(inventory_df)}')
    rows = []
    for suite, indices in inventory_df.groupby('suite').groups.items():
        rows.append({
            'suite': suite,
            'finished': sum(visibility_reviewer._scene_complete(i) for i in indices),
            'total': len(indices),
        })
    display(pd.DataFrame(rows).set_index('suite'))

Visibility finished: 9/102


,finished,total
suite,,
libero_goal,0,30
libero_object,0,30
libero_spatial,0,30
simpler_bridge,9,12


## 5. Полный интерактивный review сцен

Для каждой сцены можно:

- отметить `usable_for_slava`;
- записать заметки;
- заполнить candidate semantic slots;
- вручную отметить видимость каждого sim object в agent/wrist view.

Кнопки Previous/Next сначала сохраняют текущую форму в DataFrame. Кнопка Export JSONL атомарно записывает DataFrame в `data/task_inventory.jsonl`.

In [42]:
scene_reviewer = None
if inventory_df.empty:
    print('Scene review is unavailable until the inventory has been collected.')
else:
    scene_reviewer = InventoryReviewer(inventory_df, DATA_DIR)
    scene_reviewer.show()

После работы с формой используйте DataFrame из reviewer:

In [43]:
if scene_reviewer is None:
    print('No scene-review summary yet.')
else:
    inventory_df = scene_reviewer.df
    summary = pd.DataFrame({
        'value': [
            len(inventory_df),
            int(inventory_df.usable_for_slava.notna().sum()),
            int((inventory_df.usable_for_slava == True).sum()),
        ]
    }, index=['total', 'reviewed', 'usable'])
    display(summary)

,value
total,102
reviewed,0
usable,0


## 6. Создание object_lexicon.csv

Список строится из уникальных `raw_name` всех собранных sim objects. Существующие русские annotations не перезаписываются при повторном запуске. Технические невидимые объекты вроде `dummy_sink_target_plane` исключаются.

In [47]:
if inventory_df.empty:
    lexicon_df = pd.DataFrame()
    print('Object lexicon is unavailable until the inventory has been collected.')
else:
    lexicon_df = create_or_update_lexicon(DATA_DIR, inventory_df)
    print('Unique lexicon objects:', len(lexicon_df))
    display(lexicon_df.head(20))

Unique lexicon objects: 27


,raw_name,category_en,category_ru,color_en,color_ru,allowed_synonyms_ru,usable_v0,notes
0,akita_black_bowl,bowl,миска,black,серая,пиала,yes,
1,alphabet_soup,soup can,банка супа,blue,синяя,консервная банка,yes,
2,baked_green_cube_3cm,cube,кубик,green,зелёный,куб,yes,
3,baked_yellow_cube_3cm,cube,кубик,yellow,жёлтый,куб,yes,
4,basket,basket,корзина,beige,бежевая,контейнер,yes,
5,bbq_sauce,barbecue sauce bottle,бутылка соуса барбекю,orange,оранжевый,соус барбекю,yes,
6,bridge_carrot_generated_modified,carrot,морковь,orange,оранжевая,морковка,yes,
7,bridge_plate_objaverse_larger,plate,тарелка,yellow,желтая,блюдце,yes,
8,bridge_spoon_generated_modified,spoon,ложка,green,зелёная,ложечка,yes,
9,butter,butter package,пачка масла,red,красная,упаковка масла,yes,


## 7. Интерактивный review лексикона

`usable_v0 = no` ставьте, если объект трудно опознать на рендере или нельзя естественно и однозначно назвать по-русски.

In [48]:
lexicon_reviewer = None
if lexicon_df.empty:
    print('Lexicon review is unavailable until object_lexicon.csv has been created.')
else:
    lexicon_reviewer = LexiconReviewer(lexicon_df, DATA_DIR / 'object_lexicon.csv')
    lexicon_reviewer.show()

## 8. Финальный экспорт и sanity checks

In [49]:
if scene_reviewer is None:
    print('Nothing to export yet: inventory is empty.')
else:
    inventory_df = scene_reviewer.df
    export_inventory_dataframe(inventory_df, DATA_DIR / 'task_inventory.jsonl')
    duplicate_uids = inventory_df.task_uid[inventory_df.task_uid.duplicated()].tolist()
    missing_agent_images = [
        row.task_uid for row in inventory_df.itertuples()
        if not (DATA_DIR / row.images['agentview_rgb']).exists()
    ]
    missing_wrist_images = [
        row.task_uid for row in inventory_df.itertuples()
        if row.source['environment'] == 'LIBERO'
        and not (DATA_DIR / row.images['wrist_rgb']).exists()
    ]
    print('Rows:', len(inventory_df))
    print('Duplicate task_uid:', duplicate_uids)
    print('Missing agent images:', missing_agent_images)
    print('Missing required LIBERO wrist images:', missing_wrist_images)
    print('Saved:', DATA_DIR / 'task_inventory.jsonl')
    print('Saved:', DATA_DIR / 'object_lexicon.csv')

Rows: 102
Duplicate task_uid: []
Missing agent images: []
Missing required LIBERO wrist images: []
Saved: /Users/alexkarachun/Documents/DEV/SLAVA_dev/data/task_inventory.jsonl
Saved: /Users/alexkarachun/Documents/DEV/SLAVA_dev/data/object_lexicon.csv
